# Exploratory Data Analysis (EDA) on Keith Galli’s Electronics Store dataset

### Loading Required Libraries

In [ ]:
import os
import glob
from itertools import combinations
from collections import Counter
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

### Loading and Combining Monthly Sales Data
The sales data is provided in separate CSV files for each month of the year. So we combine them into one single data frame

In [ ]:
DATA_DIR = "/content/sample_data" #path for folder containing the list of sales data every month

all_files = glob.glob(os.path.join(DATA_DIR, "*.csv"))
print("CSV files found:", len(all_files))
all_files[:3]

dfs = []
for file in all_files:
    df = pd.read_csv(file)
    df["source_file"] = os.path.basename(file) 
    dfs.append(df)

sales_raw = pd.concat(dfs, ignore_index=True)
sales_raw.head()

sales_raw.info()


### Data Exploration
Here we check the df shape, and details.

In [ ]:
print(sales_raw.shape)
print(sales_raw.isna().sum())
print(sales_raw.duplicated().sum())

sales_raw.describe(include="all")


### Drop missing rows

In [ ]:
sales = sales_raw.copy()

sales.dropna(how="any", inplace=True)

### Remove "header rows" that appear inside the dataset

In [ ]:
sales = sales[sales["Order Date"].str.contains("Order Date") == False]

### Convert columns to correct dtypes

In [ ]:
sales["Quantity Ordered"] = pd.to_numeric(sales["Quantity Ordered"], errors="coerce")
sales["Price Each"] = pd.to_numeric(sales["Price Each"], errors="coerce")
sales["Order Date"] = pd.to_datetime(sales["Order Date"], errors="coerce")

# Removing any rows that failed conversion
sales.dropna(subset=["Quantity Ordered", "Price Each", "Order Date"], inplace=True)

# Casting quantity to int
sales["Quantity Ordered"] = sales["Quantity Ordered"].astype(int)

sales.info()

## Feature Engineering

In [ ]:
# Sales revenue
sales["Sales"] = sales["Quantity Ordered"] * sales["Price Each"]

# Month
sales["Month"] = sales["Order Date"].dt.month
sales["Month Name"] = sales["Order Date"].dt.strftime("%b")  # Jan, Feb, ...

# City extraction from address
sales["City"] = sales["Purchase Address"].str.split(",").str[1].str.strip()

# State to use along with city to avoid name clashes
sales["State"] = sales["Purchase Address"].str.split(",").str[2].str.strip().str.split(" ").str[0]

sales["City (State)"] = sales["City"] + " (" + sales["State"] + ")" # City + State label
 

sales["Hour"] = sales["Order Date"].dt.hour # Hour

sales["Day Name"] = sales["Order Date"].dt.day_name() # Day of week

sales.head()

## Descriptive Statistics

In [ ]:
sales[["Quantity Ordered", "Price Each", "Sales"]].describe()

## Q1) Best month by sales

In [ ]:
monthly_sales = sales.groupby("Month")["Sales"].sum().reset_index()
monthly_sales

### Total Sales by Month

In [ ]:
plt.figure()
plt.bar(monthly_sales["Month"], monthly_sales["Sales"])
plt.title("Total Sales by Month")
plt.xlabel("Month")
plt.ylabel("Sales Revenue ($)")
plt.xticks(monthly_sales["Month"])
plt.show()

best_row = monthly_sales.loc[monthly_sales["Sales"].idxmax()]
best_month = int(best_row["Month"])
best_revenue = float(best_row["Sales"])

print(f"Best month: {best_month} with revenue: ${best_revenue:,.2f}")

In [ ]:
month_name_map = {i: pd.Timestamp(year=2019, month=i, day=1).strftime("%b") for i in range(1, 13)}
plt.figure()
plt.bar(monthly_sales["Month"].map(month_name_map), monthly_sales["Sales"])
plt.title("Total Sales by Month")
plt.xlabel("Month")
plt.ylabel("Sales Revenue ($)")
plt.show()

## Q2) City with most products sold (Quantity)

In [ ]:
city_qty = sales.groupby("City (State)")["Quantity Ordered"].sum().sort_values(ascending=False)
city_rev = sales.groupby("City (State)")["Sales"].sum().sort_values(ascending=False)

city_qty.head(10), city_rev.head(10)

In [ ]:
plt.figure()
city_qty.plot(kind="bar")
plt.title("Total Quantity Sold by City")
plt.xlabel("City")
plt.ylabel("Quantity Ordered")
plt.xticks(rotation=45, ha="right")
plt.show()

top_city_qty = city_qty.idxmax()
print("City with most products sold (quantity):", top_city_qty, "with", int(city_qty.max()), "units")

In [ ]:
plt.figure()
city_rev.plot(kind="bar")
plt.title("Total Revenue by City")
plt.xlabel("City")
plt.ylabel("Sales Revenue ($)")
plt.xticks(rotation=45, ha="right")
plt.show()

top_city_rev = city_rev.idxmax()
print("City with highest revenue:", top_city_rev, "with $", f"{city_rev.max():,.2f}")

## Q3) Best time for ads

In [ ]:
orders_by_hour = sales.groupby("Hour")["Order ID"].count()
sales_by_hour = sales.groupby("Hour")["Sales"].sum()

orders_by_hour, sales_by_hour

In [ ]:
plt.figure()
plt.plot(orders_by_hour.index, orders_by_hour.values, marker="o")
plt.title("Number of Orders by Hour")
plt.xlabel("Hour (0-23)")
plt.ylabel("Order Count")
plt.xticks(range(0, 24))
plt.grid(True)
plt.show()

best_hour_orders = int(orders_by_hour.idxmax())
print("Peak hour by number of orders:", best_hour_orders)

In [ ]:
plt.figure()
plt.plot(sales_by_hour.index, sales_by_hour.values, marker="o")
plt.title("Total Sales Revenue by Hour")
plt.xlabel("Hour (0-23)")
plt.ylabel("Sales Revenue ($)")
plt.xticks(range(0, 24))
plt.grid(True)
plt.show()

best_hour_sales = int(sales_by_hour.idxmax())
print("Peak hour by sales revenue:", best_hour_sales)

## Q4) Products sold together

In [ ]:
# Only keep orders that appear more than once
dupe_orders = sales[sales["Order ID"].duplicated(keep=False)].copy()

# Group products per order id into a single string list
dupe_orders["Grouped"] = dupe_orders.groupby("Order ID")["Product"].transform(lambda x: ",".join(x))

dupe_unique = dupe_orders[["Order ID", "Grouped"]].drop_duplicates()

dupe_unique.head()

In [ ]:
count_pairs = Counter()

for row in dupe_unique["Grouped"]:
    products = row.split(",")
    # Count pairs (2-product combos)
    count_pairs.update(Counter(combinations(products, 2)))

# Top 10 most common pairs
top_10_pairs = count_pairs.most_common(10)
top_10_pairs

In [ ]:
for pair, cnt in top_10_pairs:
    print(f"{pair} -> {cnt}")

In [ ]:
count_triples = Counter()
for row in dupe_unique["Grouped"]:
    products = row.split(",")
    count_triples.update(Counter(combinations(products, 3)))

count_triples.most_common(10)

## Q5) Best-selling product

In [ ]:
product_qty = sales.groupby("Product")["Quantity Ordered"].sum().sort_values(ascending=False)
product_price = sales.groupby("Product")["Price Each"].mean()  # average price
product_sales = sales.groupby("Product")["Sales"].sum().sort_values(ascending=False)

product_qty.head(10), product_sales.head(10)

In [ ]:
plt.figure()
product_qty.plot(kind="bar")
plt.title("Quantity Sold per Product")
plt.xlabel("Product")
plt.ylabel("Quantity Ordered")
plt.xticks(rotation=45, ha="right")
plt.show()

best_product = product_qty.idxmax()
print("Best-selling product by units:", best_product, "with", int(product_qty.max()), "units")

In [ ]:
# Merge for analysis
prod_summary = pd.DataFrame({
    "Quantity Ordered": product_qty,
    "Avg Price": product_price,
    "Revenue": sales.groupby("Product")["Sales"].sum()
}).reset_index().rename(columns={"index": "Product"})

prod_summary.head()

#### Quantity vs Price

In [ ]:
plt.figure()
plt.scatter(prod_summary["Avg Price"], prod_summary["Quantity Ordered"])
plt.title("Quantity Ordered vs Average Price (by Product)")
plt.xlabel("Average Price ($)")
plt.ylabel("Quantity Ordered")
plt.grid(True)
plt.show()

#### Most expensive and Least expensive Products

In [ ]:
print("Most expensive products:")
display(prod_summary.sort_values("Avg Price", ascending=False).head(5))

print("\nLeast expensive products:")
display(prod_summary.sort_values("Avg Price", ascending=True).head(5))

In [ ]:
appear_count = 0
for (a, b), cnt in count_pairs.items():
    if best_product in (a, b):
        appear_count += cnt

print(f"{best_product} appears in pair-sales approx {appear_count} times (sum across pairs).")

### Extra Q6) Seasonal trends by product category (simple heuristic)

In [ ]:
def categorize_product(p: str) -> str:
    p = p.lower()
    if "phone" in p:
        return "Phones"
    if "laptop" in p:
        return "Laptops"
    if "tv" in p:
        return "TVs"
    if "headphones" in p or "earphones" in p:
        return "Audio"
    if "battery" in p or "charging" in p or "charger" in p or "cable" in p:
        return "Accessories"
    if "monitor" in p:
        return "Monitors"
    if "washer" in p or "dryer" in p:
        return "Appliances"
    return "Other"

sales["Category"] = sales["Product"].apply(categorize_product)

cat_month = sales.groupby(["Month", "Category"])["Sales"].sum().reset_index()
cat_month.head()


In [ ]:
cat_pivot = cat_month.pivot(index="Month", columns="Category", values="Sales").fillna(0)

plt.figure()
for col in cat_pivot.columns:
    plt.plot(cat_pivot.index, cat_pivot[col], marker="o", label=col)

plt.title("Seasonal Revenue Trends by Category")
plt.xlabel("Month")
plt.ylabel("Sales Revenue ($)")
plt.xticks(range(1, 13))
plt.grid(True)
plt.legend()
plt.show()

### Extra Q7) Day-of-week sales analysis

In [ ]:
dow_sales = sales.groupby("Day Name")["Sales"].sum()

dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow_sales = dow_sales.reindex(dow_order)

plt.figure()
dow_sales.plot(kind="bar")
plt.title("Sales Revenue by Day of Week")
plt.xlabel("Day")
plt.ylabel("Sales Revenue ($)")
plt.xticks(rotation=45, ha="right")
plt.show()

print("Best day by revenue:", dow_sales.idxmax(), f"(${dow_sales.max():,.2f})")